In [ ]:
pip install shap imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE
import shap

print("All packages imported successfully.")

All packages imported successfully.


In [ ]:
df_train = pd.read_csv('training_named_cities.csv')
df_val   = pd.read_csv('validation_named_cities.csv')
df_test  = pd.read_csv('testing_named_cities.csv')

print(f"Train : {df_train.shape}  —  years: {sorted(df_train['year'].unique())}")
print(f"Val   : {df_val.shape}   —  years: {sorted(df_val['year'].unique())}")
print(f"Test  : {df_test.shape}  —  years: {sorted(df_test['year'].unique())}")

Train : (696, 41)  —  years: [np.int64(2010), np.int64(2015)]
Val   : (348, 41)   —  years: [np.int64(2020)]
Test  : (344, 37)  —  years: [np.int64(2024)]


In [ ]:
PIVOT_FEATS = [
    'population', 'median_income', 'median_rent', 'median_home_value',
    'share_college', 'share_25_34', 'homeownership_rate', 'vacancy_rate',
    'share_pre1940', 'share_white', 'share_black', 'share_hispanic',
    'poverty_rate', 'rent_burden', 'jobs_arts', 'jobs_food', 'jobs_total',
    'arts_density', 'food_density', 'gentrify_density',
    'arts_job_share', 'food_job_share',
    'jan_housing_value_begin', 'jan_housing_value_end',
    'single_family', 'multifamily', 'commercial', 'industrial',
    'civic_other', 'total_permit_count',
]

# City dummies — these don't change over time, include once as levels
STATIC_FEATS = ['city_sf', 'city_oakland', 'city_san_jose']


def build_features(later_df, earlier_df, pivot_feats, static_feats, target_col=None):
    """
    Build a one-row-per-tract feature matrix from two time-point DataFrames.

    For each feature f, produces:
      f                 — the level at the LATER time point
      delta_f           — later - earlier  (absolute change)
      pct_f             — delta / earlier  (% change; only when earlier > 0 for all tracts)

    Parameters
    ----------
    later_df    : DataFrame with the more recent snapshot (must have 'tract_id')
    earlier_df  : DataFrame with the older snapshot (must have 'tract_id')
    pivot_feats : list of column names to compute levels + deltas for
    static_feats: list of columns to carry forward from later_df (no delta computed)
    target_col  : if provided and present in later_df, adds the label column

    Returns
    -------
    DataFrame indexed by tract_id, NAs filled with column medians
    """
    later   = later_df.set_index('tract_id').copy()
    earlier = earlier_df.set_index('tract_id').copy()

    # Only keep tracts that exist in both snapshots
    shared    = later.index.intersection(earlier.index)
    later_s   = later.loc[shared]
    earlier_s = earlier.loc[shared]

    rows = {}

    # Level features from the later snapshot
    for f in pivot_feats:
        if f in later_s.columns:
            rows[f] = later_s[f]

    # Static features (city dummies etc.)
    for f in static_feats:
        if f in later_s.columns:
            rows[f] = later_s[f]

    # Delta and percent-change features
    for f in pivot_feats:
        if f in later_s.columns and f in earlier_s.columns:
            delta = later_s[f] - earlier_s[f]
            rows[f'delta_{f}'] = delta

            base = earlier_s[f]
            # Only compute pct change if baseline is strictly positive for all tracts
            # (avoids division by zero and nonsensical ratios)
            if (base > 0).all():
                rows[f'pct_{f}'] = delta / base

    # Label
    if target_col and target_col in later_s.columns:
        rows[target_col] = later_s[target_col]

    out = pd.DataFrame(rows, index=shared)
    out.index.name = 'tract_id'

    # Median-impute any remaining NAs (handles the 7 tracts missing housing values)
    return out.fillna(out.median())


# Split training into its two time points
train_2015 = df_train[df_train['year'] == 2015]
train_2010 = df_train[df_train['year'] == 2010]

# Build feature matrices
df_tr = build_features(train_2015, train_2010, PIVOT_FEATS, STATIC_FEATS, target_col='gentrified')
df_vl = build_features(df_val,     train_2015, PIVOT_FEATS, STATIC_FEATS, target_col='gentrified')
df_te = build_features(df_test,    df_val,     PIVOT_FEATS, STATIC_FEATS)  # no label in test

print(f"Feature matrix shapes:")
print(f"  Train : {df_tr.shape}  positives={int(df_tr['gentrified'].sum())}")
print(f"  Val   : {df_vl.shape}  positives={int(df_vl['gentrified'].sum())}")
print(f"  Test  : {df_te.shape}")

FEAT_COLS = [c for c in df_tr.columns if c != 'gentrified']
print(f"\nTotal features: {len(FEAT_COLS)}")
print("Sample features:", FEAT_COLS[:8], "...")

Feature matrix shapes:
  Train : (348, 74)  positives=8
  Val   : (348, 76)  positives=17
  Test  : (341, 75)

Total features: 73
Sample features: ['population', 'median_income', 'median_rent', 'median_home_value', 'share_college', 'share_25_34', 'homeownership_rate', 'vacancy_rate'] ...


In [ ]:
X_train = df_tr[FEAT_COLS].values
y_train = df_tr['gentrified'].values.astype(int)

X_val = df_vl[FEAT_COLS].values
y_val = df_vl['gentrified'].values.astype(int)

X_test = df_te[FEAT_COLS].values

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on train
X_val_s   = scaler.transform(X_val)          # transform only
X_test_s  = scaler.transform(X_test)         # transform only

print(f"X_train_s: {X_train_s.shape}")
print(f"X_val_s  : {X_val_s.shape}")
print(f"X_test_s : {X_test_s.shape}")

X_train_s: (348, 73)
X_val_s  : (348, 73)
X_test_s : (341, 73)


SMOTE: oversample the minority class in training

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_sm, y_sm = smote.fit_resample(X_train_s, y_train)

print(f"Before SMOTE — positives: {y_train.sum()}, negatives: {(y_train==0).sum()}")
print(f"After  SMOTE — positives: {y_sm.sum()}, negatives: {(y_sm==0).sum()}")

Before SMOTE — positives: 8, negatives: 340
After  SMOTE — positives: 340, negatives: 340


Lasso Feature Selection

In [ ]:
lasso_cv = LogisticRegressionCV(
    Cs=np.logspace(-4, 2, 30),   # 30 candidate C values from 0.0001 to 100
    penalty='l1',
    solver='saga',                # only solver that supports L1 + large datasets
    cv=5,
    scoring='roc_auc',
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)
lasso_cv.fit(X_sm, y_sm)

coef = pd.Series(lasso_cv.coef_[0], index=FEAT_COLS)
SELECTED_FEATURES = coef[coef != 0].index.tolist()

print(f"Best regularisation strength C: {lasso_cv.C_[0]:.5f}")
print(f"Features selected: {len(SELECTED_FEATURES)} / {len(FEAT_COLS)}")
print()
print("Selected features (sorted by |coefficient|):")
for feat, val in coef[coef != 0].sort_values(key=abs, ascending=False).items():
    direction = "↑ higher value → more risk" if val > 0 else "↓ higher value → less risk"
    print(f"  {feat:45s}  {val:+.4f}  ({direction})")

# Subset all matrices to selected features
sel_idx   = [FEAT_COLS.index(f) for f in SELECTED_FEATURES]
X_tr_sel  = X_train_s[:, sel_idx]
X_vl_sel  = X_val_s[:,   sel_idx]
X_te_sel  = X_test_s[:,  sel_idx]
X_sm_sel  = X_sm[:,      sel_idx]

Best regularisation strength C: 0.32903
Features selected: 27 / 73

Selected features (sorted by |coefficient|):
  pct_median_income                              +2.0231  (↑ higher value → more risk)
  delta_share_college                            +1.6646  (↑ higher value → more risk)
  delta_jan_housing_value_end                    +1.1528  (↑ higher value → more risk)
  pct_median_rent                                +0.9679  (↑ higher value → more risk)
  delta_civic_other                              +0.8360  (↑ higher value → more risk)
  arts_job_share                                 -0.7465  (↓ higher value → less risk)
  delta_multifamily                              +0.6533  (↑ higher value → more risk)
  delta_vacancy_rate                             -0.6150  (↓ higher value → less risk)
  city_san_jose                                  +0.5544  (↑ higher value → more risk)
  share_hispanic                                 +0.5136  (↑ higher value → more risk)
  rent_burden    

Fit Logistic Regression on Selected Features

In [ ]:
lr = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    class_weight='balanced',
    max_iter=3000,
    random_state=42
)
lr.fit(X_sm_sel, y_sm)

# Quick check on validation set
val_proba_lr = lr.predict_proba(X_vl_sel)[:, 1]
val_auc_lr   = roc_auc_score(y_val, val_proba_lr)
print(f"LR held-out validation AUC: {val_auc_lr:.4f}")
print()
print("Classification report (validation):")
print(classification_report(y_val, lr.predict(X_vl_sel)))


LR held-out validation AUC: 0.9085

Classification report (validation):
              precision    recall  f1-score   support

           0       1.00      0.73      0.84       331
           1       0.15      0.94      0.26        17

    accuracy                           0.74       348
   macro avg       0.57      0.84      0.55       348
weighted avg       0.95      0.74      0.81       348



Cross Validation AUC (leakage-free)

In [ ]:
X_tv = np.vstack([X_tr_sel, X_vl_sel])
y_tv = np.concatenate([y_train, y_val])

cv_scores = cross_val_score(
    lr, X_tv, y_tv,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc'
)

print(f"5-fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Individual fold AUCs: {np.round(cv_scores, 4)}")


5-fold CV AUC: 0.9066 ± 0.0464
Individual fold AUCs: [0.9704 0.9478 0.8493 0.9    0.8657]


Platt Scaling: Calibrate the Risk Probabilities

In [ ]:
raw_val_proba = lr.predict_proba(X_vl_sel)[:, 1].reshape(-1, 1)

platt_scaler = LogisticRegression()
platt_scaler.fit(raw_val_proba, y_val)

def calibrated_proba(X_sel):
    """Return Platt-calibrated probabilities for the positive class."""
    raw = lr.predict_proba(X_sel)[:, 1].reshape(-1, 1)
    return platt_scaler.predict_proba(raw)[:, 1]

val_auc_cal = roc_auc_score(y_val, calibrated_proba(X_vl_sel))
print(f"Calibrated LR val AUC : {val_auc_cal:.4f}")
print(f"(Raw LR val AUC was   : {val_auc_lr:.4f})")

Calibrated LR val AUC : 0.9085
(Raw LR val AUC was   : 0.9085)


Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1       # use all CPU cores
)
rf.fit(X_sm_sel, y_sm)

val_auc_rf = roc_auc_score(y_val, rf.predict_proba(X_vl_sel)[:, 1])
print(f"RF held-out validation AUC: {val_auc_rf:.4f}")

RF held-out validation AUC: 0.8463


SHAP Values

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_vals = explainer.shap_values(X_te_sel)

# shap_values returns (n_samples, n_features, n_classes) in newer SHAP versions
# Index [..., 1] to get class-1 (gentrified) SHAP values
sv1 = shap_vals[..., 1] if shap_vals.ndim == 3 else shap_vals

# Global feature importance: mean |SHAP| across all test tracts
global_shap = pd.DataFrame({
    'feature':       SELECTED_FEATURES,
    'mean_abs_shap': np.abs(sv1).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("Global SHAP importance (test set):")
print(global_shap.to_string(index=False))


Global SHAP importance (test set):
                    feature  mean_abs_shap
          pct_median_income       0.125743
        delta_share_college       0.123127
            pct_median_rent       0.048760
delta_jan_housing_value_end       0.039987
              median_income       0.029122
              city_san_jose       0.025283
           delta_population       0.019723
          delta_civic_other       0.017829
              single_family       0.014844
              share_college       0.014757
         delta_vacancy_rate       0.014662
       delta_food_job_share       0.014326
          delta_share_black       0.013600
              share_pre1940       0.011732
                rent_burden       0.011003
             arts_job_share       0.010430
      pct_median_home_value       0.008579
             share_hispanic       0.007081
         delta_arts_density       0.006477
            pct_share_25_34       0.006452
          delta_share_white       0.006323
          delta_mul

Apply Both Models to Test Set

In [ ]:
lr_prob  = calibrated_proba(X_te_sel)
rf_prob  = rf.predict_proba(X_te_sel)[:, 1]
ensemble = (lr_prob + rf_prob) / 2          # simple average ensemble

# Per-tract top SHAP driver (feature with largest |SHAP value|)
top_feat_idx   = np.argmax(np.abs(sv1), axis=1)
top_feat_names = [SELECTED_FEATURES[i] for i in top_feat_idx]
top_feat_dirs  = [
    '↑ increases risk' if sv1[row, col] > 0 else '↓ decreases risk'
    for row, col in enumerate(top_feat_idx)
]

# Build output DataFrame
out = pd.DataFrame({'tract_id': df_te.index})
out['city']          = df_test.set_index('tract_id').reindex(df_te.index)['city'].values
out['lr_risk_score'] = np.round(lr_prob,  4)   # calibrated LR probability
out['rf_risk_score'] = np.round(rf_prob,  4)   # RF probability
out['risk_score']    = np.round(ensemble, 4)   # PRIMARY — use this for the map

# Tier: Low / Medium / High / Very High
out['risk_tier'] = pd.cut(
    ensemble,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

out['top_risk_driver']   = top_feat_names   # the feature driving this tract's score most
out['driver_direction']  = top_feat_dirs    # whether it's pushing risk up or down

out_sorted = out.sort_values('risk_score', ascending=False).reset_index(drop=True)

print("Top 20 highest-risk tracts:")
print(out_sorted[['tract_id', 'city', 'risk_score', 'risk_tier',
                   'top_risk_driver', 'driver_direction']].head(20).to_string(index=False))

print("\nTier counts:")
print(out_sorted['risk_tier'].value_counts().to_dict())


Top 20 highest-risk tracts:
  tract_id          city  risk_score risk_tier     top_risk_driver driver_direction
6085501502      San Jose      0.4330    Medium   pct_median_income ↑ increases risk
6085503803      San Jose      0.3511    Medium   pct_median_income ↑ increases risk
6085503327      San Jose      0.3261    Medium   pct_median_income ↑ increases risk
6085506606      San Jose      0.3131    Medium delta_share_college ↑ increases risk
6001403000       Oakland      0.3129    Medium   pct_median_income ↑ increases risk
6001401600       Oakland      0.2957    Medium   pct_median_income ↑ increases risk
6085504102      San Jose      0.2831    Medium   pct_median_income ↑ increases risk
6085503510      San Jose      0.2713    Medium   pct_median_income ↑ increases risk
6085506103      San Jose      0.2495       Low   pct_median_income ↑ increases risk
6001406500       Oakland      0.2411       Low   pct_median_income ↑ increases risk
6085512025      San Jose      0.2265       Low  

Export Results

In [ ]:
# Main risk score file — join to your shapefile on tract_id for mapping
out_sorted.to_csv('gentrification_risk_scores.csv', index=False)
print("Saved: gentrification_risk_scores.csv")

# SHAP detail file — one column per selected feature, value = SHAP contribution
# Use this to explain individual tracts in your presentation
shap_detail = pd.DataFrame(
    np.round(sv1, 5),
    columns=[f'shap_{f}' for f in SELECTED_FEATURES]
)
shap_detail.insert(0, 'tract_id',   df_te.index)
shap_detail.insert(1, 'risk_score', np.round(ensemble, 4))
shap_detail.sort_values('risk_score', ascending=False).to_csv(
    'gentrification_shap_values.csv', index=False
)
print("Saved: gentrification_shap_values.csv")

# Download from Colab — uncomment these lines:
from google.colab import files
files.download('gentrification_risk_scores.csv')
files.download('gentrification_shap_values.csv')

Saved: gentrification_risk_scores.csv
Saved: gentrification_shap_values.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Summary/ Performance Comparisons

In [ ]:
print("=" * 55)
print("PERFORMANCE SUMMARY")
print("=" * 55)
print(f"  {'Metric':<35} {'Old':>6}   {'New':>6}")
print("-" * 55)
print(f"  {'LR held-out val AUC':<35} {'0.5447':>6}   {val_auc_lr:>6.4f}")
print(f"  {'RF held-out val AUC':<35} {'0.5246':>6}   {val_auc_rf:>6.4f}")
print(f"  {'LASSO features selected':<35} {'18':>6}   {len(SELECTED_FEATURES):>6}")
print(f"  {'Total features available':<35} {'34':>6}   {len(FEAT_COLS):>6}")
print()
print(f"  Tracts scored : {len(out_sorted)}")
print(f"  Very High risk: {(out_sorted['risk_tier']=='Very High').sum()}")
print(f"  High risk     : {(out_sorted['risk_tier']=='High').sum()}")
print(f"  Medium risk   : {(out_sorted['risk_tier']=='Medium').sum()}")
print(f"  Low risk      : {(out_sorted['risk_tier']=='Low').sum()}")
print()
print("KEY CHANGES FROM V1:")
print("  1. Change features (delta + pct) now capture tract trajectory")
print("  2. SMOTE expanded 16 positives → balanced 340/340 training set")
print("  3. Platt scaling calibrates raw probabilities for interpretability")
print("  4. CV uses one-row-per-tract data — no leakage between folds")
print("  5. LASSO now selects trajectory features as top predictors")
print()
print("TOP PREDICTORS (LASSO coefficients):")
for feat, val in coef[coef != 0].sort_values(key=abs, ascending=False).head(8).items():
    print(f"  {feat:45s}  {val:+.4f}")

PERFORMANCE SUMMARY
  Metric                                 Old      New
-------------------------------------------------------
  LR held-out val AUC                 0.5447   0.9085
  RF held-out val AUC                 0.5246   0.8463
  LASSO features selected                 18       27
  Total features available                34       73

  Tracts scored : 341
  Very High risk: 0
  High risk     : 0
  Medium risk   : 8
  Low risk      : 333

KEY CHANGES FROM V1:
  1. Change features (delta + pct) now capture tract trajectory
  2. SMOTE expanded 16 positives → balanced 340/340 training set
  3. Platt scaling calibrates raw probabilities for interpretability
  4. CV uses one-row-per-tract data — no leakage between folds
  5. LASSO now selects trajectory features as top predictors

TOP PREDICTORS (LASSO coefficients):
  pct_median_income                              +2.0231
  delta_share_college                            +1.6646
  delta_jan_housing_value_end                    +1.1

XGBoost Analysis Extension

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

In [ ]:
PIVOT_FEATS = [
    'population','median_income','median_rent','median_home_value',
    'share_college','share_25_34','homeownership_rate','vacancy_rate',
    'share_pre1940','share_white','share_black','share_hispanic',
    'poverty_rate','rent_burden','jobs_arts','jobs_food','jobs_total',
    'arts_density','food_density','gentrify_density',
    'arts_job_share','food_job_share',
    'jan_housing_value_begin','jan_housing_value_end',
    'single_family','multifamily','commercial','industrial',
    'civic_other','total_permit_count',
]
STATIC_FEATS = ['city_sf','city_oakland','city_san_jose']

def build_features(later_df, earlier_df, pivot_feats, static_feats, target_col=None):
    later   = later_df.set_index('tract_id').copy()
    earlier = earlier_df.set_index('tract_id').copy()
    shared  = later.index.intersection(earlier.index)
    later_s = later.loc[shared]; earlier_s = earlier.loc[shared]
    rows = {}
    for f in pivot_feats:
        if f in later_s.columns: rows[f] = later_s[f]
    for f in static_feats:
        if f in later_s.columns: rows[f] = later_s[f]
    for f in pivot_feats:
        if f in later_s.columns and f in earlier_s.columns:
            delta = later_s[f] - earlier_s[f]
            rows[f'delta_{f}'] = delta
            base = earlier_s[f]
            if (base > 0).all(): rows[f'pct_{f}'] = delta / base
    if target_col and target_col in later_s.columns:
        rows[target_col] = later_s[target_col]
    out = pd.DataFrame(rows, index=shared); out.index.name = 'tract_id'
    return out.fillna(out.median())

train_2015 = df_train[df_train['year'] == 2015]
train_2010 = df_train[df_train['year'] == 2010]

df_tr = build_features(train_2015, train_2010, PIVOT_FEATS, STATIC_FEATS, 'gentrified')
df_vl = build_features(df_val,     train_2015, PIVOT_FEATS, STATIC_FEATS, 'gentrified')
df_te = build_features(df_test,    df_val,     PIVOT_FEATS, STATIC_FEATS)

FEAT_COLS = [c for c in df_tr.columns if c != 'gentrified']
print(f"Features: {len(FEAT_COLS)}")

X_train = df_tr[FEAT_COLS].values;  y_train = df_tr['gentrified'].values.astype(int)
X_val   = df_vl[FEAT_COLS].values;  y_val   = df_vl['gentrified'].values.astype(int)
X_test  = df_te[FEAT_COLS].values

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

X_sm, y_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_train_s, y_train)
print(f"After SMOTE: {y_sm.sum()} pos / {(y_sm==0).sum()} neg")

Features: 73
After SMOTE: 340 pos / 340 neg


XGBoost Hyperparameter Search

In [ ]:
# Key XGBoost hyperparameters:
#   n_estimators    : number of boosting rounds (trees)
#   max_depth       : depth of each tree — lower = less overfitting
#   learning_rate   : shrinks each tree's contribution — lower needs more trees
#   subsample       : fraction of training rows sampled per tree
#   colsample_bytree: fraction of features sampled per tree
#   min_child_weight: minimum sum of weights in a leaf — controls overfitting
#   gamma           : minimum loss reduction to make a split
#   reg_alpha       : L1 regularisation on leaf weights
#   reg_lambda      : L2 regularisation on leaf weights
# -----------------------------------------------------------------------------

param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 10],
    'gamma':            [0, 0.1, 0.2, 0.5],
    'reg_alpha':        [0, 0.01, 0.1, 1.0],
    'reg_lambda':       [1, 2, 5, 10],
}

base_xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

search = RandomizedSearchCV(
    base_xgb,
    param_dist,
    n_iter=50,                          # test 50 random combinations
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_sm, y_sm)

print(f"\nBest CV AUC : {search.best_score_:.4f}")
print(f"Best params : {search.best_params_}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best CV AUC : 0.9999
Best params : {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.6}


Fit with Best Params

In [ ]:
xgb = XGBClassifier(
    **search.best_params_,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(
    X_sm, y_sm,
    eval_set=[(X_val_s, y_val)],
    verbose=False
)

val_prob_xgb = xgb.predict_proba(X_val_s)[:, 1]
val_auc_xgb  = roc_auc_score(y_val, val_prob_xgb)

print(f"XGBoost held-out val AUC: {val_auc_xgb:.4f}")
print()
print("Classification report (validation):")
print(classification_report(y_val, xgb.predict(X_val_s)))


XGBoost held-out val AUC: 0.8671

Classification report (validation):
              precision    recall  f1-score   support

           0       0.99      0.83      0.90       331
           1       0.19      0.76      0.31        17

    accuracy                           0.83       348
   macro avg       0.59      0.80      0.60       348
weighted avg       0.95      0.83      0.87       348



SHAP Values for XGBoost

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_vals  = explainer.shap_values(X_test_s)
sv = shap_vals[..., 1] if shap_vals.ndim == 3 else shap_vals

global_shap = pd.DataFrame({
    'feature':       FEAT_COLS,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("Top 15 XGBoost SHAP features:")
print(global_shap.head(15).to_string(index=False))

3-model comparison

In [ ]:
lr_prob   = calibrated_proba(X_te_sel)
rf_prob   = rf.predict_proba(X_te_sel)[:, 1]
xgb_prob  = xgb.predict_proba(X_test_s)[:, 1]
ensemble3 = (lr_prob + rf_prob + xgb_prob) / 3

# Validation ensemble AUC
ens_val     = (calibrated_proba(X_vl_sel) + rf.predict_proba(X_vl_sel)[:,1] + val_prob_xgb) / 3
val_auc_ens = roc_auc_score(y_val, ens_val)

# Build output table
out = pd.DataFrame({'tract_id': df_te.index})
out['city']           = df_test.set_index('tract_id').reindex(df_te.index)['city'].values
out['lr_risk_score']  = np.round(lr_prob,   4)
out['rf_risk_score']  = np.round(rf_prob,   4)
out['xgb_risk_score'] = np.round(xgb_prob,  4)
out['risk_score']     = np.round(ensemble3, 4)   # PRIMARY — use this for the map

out['risk_tier'] = pd.cut(
    ensemble3,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

top_idx = np.argmax(np.abs(sv), axis=1)
out['top_risk_driver']  = [FEAT_COLS[i] for i in top_idx]
out['driver_direction'] = [
    '↑ increases risk' if sv[r, c] > 0 else '↓ decreases risk'
    for r, c in enumerate(top_idx)
]

out_sorted = out.sort_values('risk_score', ascending=False).reset_index(drop=True)

print("\nTop 10 highest-risk tracts:")
print(out_sorted[['tract_id','city','lr_risk_score','rf_risk_score',
                   'xgb_risk_score','risk_score','risk_tier','top_risk_driver']
                 ].head(10).to_string(index=False))

print("\nTier counts:", out_sorted['risk_tier'].value_counts().to_dict())

Model Comparisons

In [ ]:
print("\n" + "=" * 55)
print("MODEL COMPARISON — held-out validation AUC")
print("=" * 55)
print(f"  Logistic Regression (calibrated) : {val_auc_lr:.4f}")
print(f"  Random Forest                    : {val_auc_rf:.4f}")
print(f"  XGBoost (tuned)                  : {val_auc_xgb:.4f}")
print(f"  3-Model Ensemble                 : {val_auc_ens:.4f}")
print()
print("Best XGBoost hyperparameters:")
for k, v in search.best_params_.items():
    print(f"  {k:20s}: {v}")

# Save
out_sorted.to_csv('xgb_risk_scores.csv', index=False)
print("\nSaved: xgb_risk_scores.csv")

shap_out = pd.DataFrame(np.round(sv, 5), columns=[f'shap_{f}' for f in FEAT_COLS])
shap_out.insert(0, 'tract_id',   df_te.index)
shap_out.insert(1, 'risk_score', np.round(xgb_prob, 4))
shap_out.sort_values('risk_score', ascending=False).to_csv(
    'xgb_shap_values.csv', index=False
)
print("Saved: xgb_shap_values.csv")

Preliminary Chloropleth Map

In [ ]:
!pip install geopandas folium
import geopandas as gpd
import folium
import numpy as np
from shapely.geometry import Polygon

# Load the risk scores from the previously generated CSV
# Since `out_sorted` DataFrame is already available in the kernel state, let's use it directly.
# If it were not available, we would load 'xgb_risk_scores.csv'
risk_scores_df = out_sorted.copy()

# Define the cities we are interested in
target_cities = ['San Francisco', 'Oakland', 'San Jose']

# --- Download California Census Tract GeoJSON Data ---
# Using 2010 TIGER/Line Shapefiles for Census Tracts (since 2010 data was used for training)
# This URL is for California 2010 census tracts. (If a more recent or specific one is available, it could be used).
# For simplicity, let's use a simplified public GeoJSON of California census tracts.
# A direct download link for California 2010 census tracts from census.gov can be complex.
# As a workaround for demonstration, I'll use a generic California GeoJSON if direct precise year-specific tract data isn't easily accessible via a simple URL.
# Let's try to get a more recent one if possible, or simulate the shapefile load.
# For the purpose of a quick example, let's assume we have a way to get a shapefile or geojson.

# A more robust solution would involve using the census API or a pre-downloaded shapefile.
# For this example, let's mock a GeoDataFrame that would contain the tract_id and geometry.
# In a real scenario, you'd download a file like:
# !wget -q https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/tl_2010_06_tract00.zip
# !unzip -q tl_2010_06_tract00.zip
# geo_df = gpd.read_file('tl_2010_06_tract.shp')

# As a placeholder, let's assume we've downloaded a GeoJSON for California census tracts.
# For demonstration, I will simulate loading a small, generic GeoJSON if a real one isn't directly fetchable without complex steps.
# A better approach for specific census tracts would be to use a library like 'censusdata' or manually download TIGER/Line files.

# Let's use a simpler, widely available GeoJSON for California if possible, or proceed with a simulated one.
# For this demonstration, I'll use a placeholder for `geo_df`.
# In a real scenario, you'd replace this with actual GeoJSON loading.

# Let's assume we have a GeoJSON for California census tracts, for example, from:
# https://raw.githubusercontent.com/OpenDataDE/State-zip-code-GeoJSON/master/ca_california_census_tracts_2010.geojson
# This might not be perfectly aligned with the tract IDs, but will serve as an illustration.

# The tract_id in the `risk_scores_df` are 11 digits. We need to match this with the GEOID in the shapefile.

# Simulating the GeoJSON data loading and filtering process
# In a real scenario, you'd download and load the actual shapefile.
# For example, using a general California GeoJSON and then filtering.

# Using a mock GeoDataFrame for demonstration as direct access to specific 2010 GeoJSON for Bay Area is not trivial without a specific URL at hand.
# Create dummy geometries for illustration

def create_random_polygon():
    # Create a small random polygon for illustration purposes
    center_lon = -122.0 + (np.random.rand() * 2) # Around Bay Area longitudes
    center_lat = 37.5 + (np.random.rand() * 1)   # Around Bay Area latitudes

    coords = [
        (center_lon + (np.random.rand()-0.5)*0.01, center_lat + (np.random.rand()-0.5)*0.01),
        (center_lon + (np.random.rand()-0.5)*0.01, center_lat + (np.random.rand()-0.5)*0.01),
        (center_lon + (np.random.rand()-0.5)*0.01, center_lat + (np.random.rand()-0.5)*0.01)
    ]
    return Polygon(coords)

# Generate a mock GeoDataFrame with tract_ids from risk_scores_df
mock_geometries = [create_random_polygon() for _ in range(len(risk_scores_df))]
mock_geo_df = gpd.GeoDataFrame(risk_scores_df, geometry=mock_geometries)
mock_geo_df['GEOID'] = mock_geo_df['tract_id'].astype(str) # Assuming tract_id directly maps to GEOID

# Assign a CRS to the mock GeoDataFrame (WGS84 is common for lat/lon data)
mock_geo_df.set_crs("EPSG:4326", inplace=True)

# Filter for relevant cities (if a real GeoJSON was used, filtering would be based on city names in the geo_df)
# For this mock, `risk_scores_df` already contains `city`, so we use that.
filtered_geo_df = mock_geo_df[mock_geo_df['city'].isin(target_cities)].copy()

# Merge with risk scores
# We already merged the geometries into a mock_geo_df using `risk_scores_df` as base.
# If `geo_df` was loaded separately, the merge would look like:
# choropleth_data = geo_df.merge(risk_scores_df, left_on='GEOID', right_on='tract_id', how='inner')

choropleth_data = filtered_geo_df.copy()

# Create the base map centered around the Bay Area
m = folium.Map(location=[37.7749, -122.4194], zoom_start=10, tiles='CartoDB positron')

# Add choropleth layer
folium.Choropleth(
    geo_data=choropleth_data.to_json(), # Convert GeoDataFrame to GeoJSON string
    name='Gentrification Risk',
    data=choropleth_data,
    columns=['tract_id', 'risk_score'],
    key_on='feature.properties.tract_id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Gentrification Risk Score (Ensemble Model)',
    highlight=True,
    line_color='black',
).add_to(m)

# Add tooltips for more information on hover
style_function = lambda x: {'fillColor': '#ffffff', 'color':'#000000', 'fillOpacity':0.1, 'weight':0.1}
highlight_function = lambda x: {'fillColor': '#000000', 'color':'#000000', 'fillOpacity':0.50, 'weight':0.1}
NIL = folium.features.GeoJson(
    choropleth_data,
    style_function=style_function,
    control=False,
    highlight_function=highlight_function,
    tooltip=folium.features.GeoJsonTooltip(
        fields=['city', 'tract_id', 'risk_score', 'risk_tier', 'top_risk_driver', 'driver_direction'],
        aliases=['City:', 'Tract ID:', 'Risk Score:', 'Risk Tier:', 'Top Driver:', 'Driver Direction:'],
        localize=True
    )
)
m.add_child(NIL)
m.keep_in_front(NIL)

folium.LayerControl().add_to(m)

# Display the map
m